In [2]:
import os, queue, threading, contextlib, sounddevice as sd
from dotenv import load_dotenv
from deepgram import DeepgramClient
from deepgram.speak.v1.types import SpeakV1Text

load_dotenv()
dg = DeepgramClient(api_key=os.environ["DEEPGRAM_API_KEY"])
RATE = int(sd.query_devices(kind="input")["default_samplerate"])

# Names, jargon, product terms. Biggest accuracy lever of anything tested here -
# on a hard sentence this took flux from 9.1% to 0% word error, nova-3 from 13.6% to 4.5%.
KEYTERMS = ["Kubernetes", "Priya", "Postgres", "standup"]

# JARVIS: British baritone, a touch under normal pace. aura-2-draco-en is the only
# British masculine aura-2 voice - aura-helios-en is the other British male, but it's
# aura-1 and rejects `speed`. Valid speed range is 0.7 - 1.5; outside that the API 400s.
VOICE, SPEED = "aura-2-draco-en", 0.9

OUT = sd.RawOutputStream(samplerate=24000, channels=1, dtype="int16", latency="low")
OUT.start()

_tail = b""
def play(chunk):
    global _tail                     # chunk boundaries don't respect 16-bit frames
    buf = _tail + chunk
    cut = len(buf) - len(buf) % 2
    try:
        OUT.write(buf[:cut])
    except Exception:
        pass                         # device swapped mid-utterance; keep the pump alive
    _tail = buf[cut:]

def use_output(match=None):
    """Move playback to another device. An open stream never follows the system
    default, and PortAudio snapshots the device list when it loads - so AirPods
    paired after the kernel started are invisible until this re-scans."""
    global OUT
    try:
        OUT.abort(); OUT.close()
    except Exception:
        pass
    sd._terminate(); sd._initialize()
    outs = [(i, d["name"]) for i, d in enumerate(sd.query_devices())
            if d["max_output_channels"] > 0]
    dev = None
    if match:
        hits = [i for i, n in outs if match.lower() in n.lower()]
        if not hits:
            raise LookupError(f"no output device matching {match!r}. "
                              f"Connected: {[n for _, n in outs]}")
        dev = hits[0]
    OUT = sd.RawOutputStream(samplerate=24000, channels=1, dtype="int16",
                             latency="low", device=dev)
    OUT.start()
    print("output ->", sd.query_devices(OUT.device)["name"])

RATE

24000

In [ ]:
# STT 1 - nova-3, waits for a pause before showing anything. Accurate, ~2.1s behind you.
PAUSE_MS = 800

q = queue.Queue()
with dg.listen.v1.connect(
    model="nova-3", encoding="linear16", sample_rate=RATE, channels=1,
    interim_results=True, punctuate=True, smart_format=True,
    endpointing=PAUSE_MS, utterance_end_ms=1000,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        line = []
        try:
            for m in sock:
                if m.type != "Results":
                    continue
                text = m.channel.alternatives[0].transcript
                if m.is_final:
                    if text:
                        line.append(text)
                    if m.speech_final and line:
                        print(" ".join(line).ljust(100))
                        line = []
                elif text:
                    print(" ".join(line + [text])[:100].ljust(100), end="\r", flush=True)
        except KeyboardInterrupt:
            q.put(None)

listening - interrupt the cell to stop



In [ ]:
# THIS IS BEST AS OF NOW
# STT 2 - flux, predicts when your turn ends. Fast, but loose on names/jargon
# unless you give it keyterms (see STT 4).
EAGER, EOT = 0.3, 0.5

q = queue.Queue()
with dg.listen.v2.connect(
    model="flux-general-en", encoding="linear16", sample_rate=RATE,
    eager_eot_threshold=EAGER, eot_threshold=EOT, eot_timeout_ms=2000,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        try:
            for m in sock:
                ev = getattr(m, "event", None)
                if ev == "EndOfTurn":
                    print(m.transcript.ljust(100))
                elif ev in ("StartOfTurn", "Update", "EagerEndOfTurn") and m.transcript:
                    print(m.transcript[:100].ljust(100), end="\r", flush=True)
        except KeyboardInterrupt:
            q.put(None)

listening - interrupt the cell to stop

Otoconjie.                                                                                          
The mitochondria is the path house of the cell.                                                     
The mitochondria is the powerhouse of the cell.                                                     
Interesting.                                                                                        


In [4]:
# STT 3 - nova-3 accuracy at roughly half the wait (0.80s vs 2.08s measured).
# Two separate signals instead of one: each phrase prints the moment it is final,
# and the line only breaks on UtteranceEnd - a real pause, not just a short gap.
PAUSE_MS = 150     # how fast a phrase finalises (lower = text lands sooner)
BREAK_MS = 1000    # silence that counts as a conversational break -> new line

q = queue.Queue()
with dg.listen.v1.connect(
    model="nova-3", encoding="linear16", sample_rate=RATE, channels=1,
    interim_results=True, punctuate=True, smart_format=True,
    endpointing=PAUSE_MS, utterance_end_ms=BREAK_MS, keyterm=KEYTERMS,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        dirty = False
        try:
            for m in sock:
                if m.type == "UtteranceEnd":
                    if dirty:
                        print()
                        dirty = False
                elif m.type == "Results" and m.is_final:
                    text = m.channel.alternatives[0].transcript
                    if text:
                        print(text + " ", end="", flush=True)
                        dirty = True
        except KeyboardInterrupt:
            q.put(None)

listening - interrupt the cell to stop

Opt in two is good speed, but it's more accuracy is a bit of a good stay. 
Can we get options three and four for both outs? 
Vout. Moots. 
Option two is fast, not very accurate. And option one is good accuracy, but a bit too slow. 


In [5]:
# STT 4 - flux with keyterms. Most accurate of the four (0% word error on the test
# sentence vs 9.1% without them), ~1.9s behind you. Add your own vocabulary to KEYTERMS.
EAGER, EOT = 0.3, 0.5

q = queue.Queue()
with dg.listen.v2.connect(
    model="flux-general-en", encoding="linear16", sample_rate=RATE,
    eager_eot_threshold=EAGER, eot_threshold=EOT, eot_timeout_ms=2000,
    keyterm=KEYTERMS,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        try:
            for m in sock:
                ev = getattr(m, "event", None)
                if ev == "EndOfTurn":
                    print(m.transcript.ljust(100))
                elif ev in ("StartOfTurn", "Update", "EagerEndOfTurn") and m.transcript:
                    print(m.transcript[:100].ljust(100), end="\r", flush=True)
        except KeyboardInterrupt:
            q.put(None)

listening - interrupt the cell to stop

Option two has good speed, it's more effective series a bit of a constraint.                        
I mean, you're often paying for for both rates.                                                     
I find that takes the speech slow on start up, but they've been speaking. Not sure if you can do much about that.


In [4]:
# TTS 1 - one-shot REST, plays chunks as they arrive. ~0.84s to first sound,
# most of which is setting up the connection.
def say(text, model=VOICE):
    for chunk in dg.speak.v1.audio.generate(
            text=text, model=model, encoding="linear16", speed=SPEED,
            container="none",        # default is wav - the header clicks through the speakers
            sample_rate=24000):
        play(chunk)

say("Good evening, sir. This is the one shot version.")

In [5]:
# TTS 2 - websocket, new connection each call. Takes text in pieces, so you can
# feed LLM tokens. ~0.80s to first sound.
def say_stream(pieces, model=VOICE):
    with dg.speak.v1.connect(model=model, encoding="linear16",
                             sample_rate=24000, speed=SPEED) as sock:
        def feed():
            for p in pieces:
                sock.send_text(SpeakV1Text(type="Speak", text=p))
            sock.send_flush()
        threading.Thread(target=feed, daemon=True).start()
        for m in sock:
            if isinstance(m, bytes):
                play(m)
            elif getattr(m, "type", None) == "Flushed":
                break

say_stream(w + " " for w in "Hey Marcus, this is the one shot version..".split())

In [15]:
# TTS 3 - the startup fix. Nearly all of that 0.8s was the handshake, not the model,
# so hold the socket open: 0.22s per line, measured on your key. Run this cell once.
SPEAKER, SOCK, MUTED, FLUSHES = contextlib.ExitStack(), None, False, 0

def _pump(sock):
    global FLUSHES
    try:
        for m in sock:
            if isinstance(m, bytes):
                if not MUTED and sock is SOCK:
                    play(m)            # blocking, so by the time Flushed lands below,
            elif getattr(m, "type", None) == "Flushed":
                FLUSHES += 1           # every chunk before it is already out the speaker
    except Exception:
        pass                          # socket closed, or replaced by a voice switch

def connect_speaker(voice=VOICE, speed=SPEED):
    """(Re)open the socket. The model is fixed at handshake, so changing voice
    means reconnecting - that's what the voice picker cell calls."""
    global SPEAKER, SOCK, MUTED
    SPEAKER.close()                   # drop the previous socket, if any
    SPEAKER, MUTED = contextlib.ExitStack(), False
    kw = dict(model=voice, encoding="linear16", sample_rate=24000)
    if voice.startswith("aura-2-"):
        kw["speed"] = speed           # aura-1 voices 400 if you send speed at all
    SOCK = SPEAKER.enter_context(dg.speak.v1.connect(**kw))
    threading.Thread(target=_pump, args=(SOCK,), daemon=True).start()

def say_now(text):
    global MUTED
    MUTED = False
    SOCK.send_text(SpeakV1Text(type="Speak", text=text))
    SOCK.send_flush()

def stop():
    global MUTED                      # don't OUT.abort() here - killing the stream from
    MUTED = True                      # another thread takes PortAudio down with it
    SOCK.send_clear()

connect_speaker()
say_now("Good evening, sir. All systems are online and running at optimal capacity.")

In [47]:
# TTS 4 - same open socket, fed a piece at a time: lowest latency AND token streaming.
# This is the one to use behind an LLM. stop() from TTS 3 works here too for barge-in.
TOKENS_VOICE = "aura-2-jupiter-en"     # expressive, knowledgeable, baritone
TOKENS_SPEED = 1.2                     # brisker; 1.5 is the ceiling, 1.6 gets a 400
connect_speaker(TOKENS_VOICE, TOKENS_SPEED)   # shared socket - re-run TTS 3 for draco at 0.9

def say_tokens(pieces):
    global MUTED
    MUTED = False
    for p in pieces:
        SOCK.send_text(SpeakV1Text(type="Speak", text=p))
    SOCK.send_flush()

say_tokens(w + " " for w in
           "May I say how refreshing it is to finally see you in a video with your clothing on sir".split())

In [38]:
#jupiter, mars, neptune

# ---- VOICE PICKER - paste a name below, run this cell, listen ----------------
# Reconnects the socket that TTS 3 (say_now) and TTS 4 (say_tokens) both use.

VOICE = "aura-2-jupiter-en"
SPEED = 1.2                   # 0.7 - 1.5 only, and aura-2 voices only

connect_speaker(VOICE, SPEED)
say_now("Good evening, sir. All systems are online and running at optimal capacity.")

# British masculine
#   aura-2-draco-en        warm, approachable, trustworthy, baritone
#   aura-helios-en         postive, comfortable, polite  [aura-1: ignores SPEED]
# British feminine
#   aura-2-pandora-en      smooth, calm, melodic, breathy
#   aura-athena-en         smooth, calm, professional  [aura-1: ignores SPEED]
# Irish masculine
#   aura-angus-en          casual, friendly, patient  [aura-1: ignores SPEED]
# Australian masculine
#   aura-2-hyperion-en     caring, warm, empathetic
# Australian feminine
#   aura-2-theia-en        expressive, polite, sincere
# American masculine
#   aura-2-apollo-en       confident, comfortable, casual
#   aura-2-arcas-en        natural, smooth, clear, comfortable
#   aura-2-aries-en        warm, energetic, caring
#   aura-2-atlas-en        enthusiastic, confident, approachable, friendly
#   aura-2-hermes-en       expressive, engaging, professional
#   aura-2-jupiter-en      expressive, knowledgeable, baritone
#   aura-2-mars-en         smooth, patient, trustworthy, baritone
#   aura-2-neptune-en      professional, patient, polite
#   aura-2-odysseus-en     calm, smooth, comfortable, professional
#   aura-2-orion-en        approachable, comfortable, calm, polite
#   aura-2-orpheus-en      professional, clear, confident, trustworthy
#   aura-2-pluto-en        smooth, calm, empathetic, baritone
#   aura-2-saturn-en       knowledgeable, confident, baritone
#   aura-2-zeus-en         deep, trustworthy, smooth
#   aura-arcas-en          natural, smooth, clear  [aura-1: ignores SPEED]
#   aura-orion-en          approachable, comfortable, calm  [aura-1: ignores SPEED]
#   aura-orpheus-en        clear, trustworthy, professional  [aura-1: ignores SPEED]
#   aura-perseus-en        expressive, melodic, charismatic  [aura-1: ignores SPEED]
#   aura-zeus-en           deep, trustworthy, smooth  [aura-1: ignores SPEED]
# American feminine
#   aura-2-andromeda-en    casual, expressive, comfortable
#   aura-2-asteria-en      clear, confident, knowledgeable, energetic
#   aura-2-athena-en       calm, smooth, professional
#   aura-2-aurora-en       cheerful, expressive, energetic
#   aura-2-callista-en     clear, energetic, professional, smooth
#   aura-2-cora-en         smooth, melodic, caring
#   aura-2-cordelia-en     approachable, warm, polite
#   aura-2-delia-en        casual, friendly, cheerful, breathy
#   aura-2-electra-en      professional, engaging, knowledgeable
#   aura-2-harmonia-en     empathetic, clear, calm, confident
#   aura-2-helena-en       caring, natural, positive, friendly
#   aura-2-hera-en         smooth, warm, professional
#   aura-2-iris-en         cheerful, positive, approachable
#   aura-2-janus-en        southern, smooth, trustworthy
#   aura-2-juno-en         natural, engaging, melodic, breathy
#   aura-2-luna-en         friendly, natural, engaging
#   aura-2-minerva-en      positive, friendly, natural
#   aura-2-ophelia-en      expressive, enthusiastic, cheerful
#   aura-2-phoebe-en       energetic, warm, casual
#   aura-2-selene-en       expressive, engaging, energetic
#   aura-2-thalia-en       clear, confident, energetic, enthusiastic
#   aura-2-vesta-en        natural, expressive, patient, empathetic
#   aura-asteria-en        clear, confident, knowledgeable  [aura-1: ignores SPEED]
#   aura-hera-en           deep, smooth, warm  [aura-1: ignores SPEED]
#   aura-luna-en           friendly, natural, engaging  [aura-1: ignores SPEED]
#   aura-stella-en         raspy, engaging, cheerful  [aura-1: ignores SPEED]
# Filipino feminine
#   aura-2-amalthea-en     engaging, natural, cheerful
# total English voices: 53
# other languages available: de, es, fr, it, ja, nl
# print the full list incl. other languages:
#   import urllib.request, json
#   r = urllib.request.Request("https://api.deepgram.com/v1/models",
#           headers={"Authorization": "Token " + os.environ["DEEPGRAM_API_KEY"]})
#   print([m["canonical_name"] for m in json.load(urllib.request.urlopen(r))["tts"]])

In [4]:
# CONVERSATION - STT 2 (flux) -> Cerebras gpt-oss-120b -> TTS 4 (jupiter).
# Run cell 0 and TTS 3 first. Interrupt the cell to hang up.
import json, time, httpx

# "Respond shortly" alone still returns markdown lists and "≈238,900 km", which the
# voice reads out literally. The rest of this keeps it speakable.
SYSTEM = ("Respond shortly. You are a voice assistant and your reply is read aloud: "
          "answer in one or two plain spoken sentences. Never use markdown, lists, "
          "bullets, asterisks, headings, or emoji. Write numbers and units as words.")
MODEL  = "gpt-oss-120b"
LLM    = httpx.Client(base_url="https://api.cerebras.ai/v1", timeout=30,
                      headers={"Authorization": f"Bearer {os.environ['CEREBUS']}"})

connect_speaker("aura-2-jupiter-en", 1.2)      # same voice/speed as TTS 4
history  = [{"role": "system", "content": SYSTEM}]
SPEAKING = False

def reply(said_to_it):
    """Pipe tokens straight from Cerebras into the open TTS socket, flushing at each
    sentence end so it starts talking on sentence one instead of the whole answer."""
    global SPEAKING, FLUSHES
    history.append({"role": "user", "content": said_to_it})
    SPEAKING = True
    before, flushes, answer = FLUSHES, 0, ""
    with LLM.stream("POST", "/chat/completions", json={
            "model": MODEL, "messages": history, "stream": True,
            # gpt-oss reasons before answering and those tokens come out of the same
            # budget - at effort "high" they can eat it entirely and return nothing.
            "reasoning_effort": "low", "max_completion_tokens": 400,
            "temperature": 0.6}) as r:
        if r.status_code != 200:
            print(f"  [cerebras {r.status_code}] {r.read().decode()[:160]}")  # 429 = per-minute cap
            SPEAKING = False
            return
        for line in r.iter_lines():
            if not line.startswith("data: "):
                continue
            body = line[6:]
            if body == "[DONE]":
                break
            tok = json.loads(body)["choices"][0]["delta"].get("content") or ""
            if not tok:
                continue
            answer += tok
            SOCK.send_text(SpeakV1Text(type="Speak", text=tok))
            if tok.rstrip()[-1:] in ".!?":
                SOCK.send_flush(); flushes += 1
    SOCK.send_flush(); flushes += 1                  # trailing text + a done marker
    history.append({"role": "assistant", "content": answer})
    print(f"  jarvis: {answer.strip() or '[empty - reasoning used the whole budget]'}")

    deadline = time.time() + 30                      # wait for the audio to actually finish
    while FLUSHES - before < flushes and time.time() < deadline:
        time.sleep(0.02)
    time.sleep(0.25)                                 # device buffer tail
    SPEAKING = False

q = queue.Queue()
with dg.listen.v2.connect(
    model="flux-general-en", encoding="linear16", sample_rate=RATE,
    eager_eot_threshold=0.3, eot_threshold=0.5, eot_timeout_ms=2000,
    # keyterm=KEYTERMS,                    # uncomment for STT 4 accuracy, same latency
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()

    for _ in range(40):                 # burst 2s of silence before opening the mic. Costs
        q.put(bytes(RATE // 20 * 2))    # ~0.1s and makes the first thing you say answer in
                                        # 0.15s instead of 1.05s - measured A/B, 7x.
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda d, *_: q.put(bytes(len(d)) if SPEAKING else bytes(d))):
        print("say something - interrupt the cell to hang up\n")   # mic feeds silence while
        try:                                                        # it talks, or it hears itself
            for m in sock:
                if getattr(m, "event", None) == "EndOfTurn" and m.transcript.strip():
                    print(f"you:    {m.transcript}")
                    reply(m.transcript)
        except KeyboardInterrupt:
            q.put(None)

NameError: name 'connect_speaker' is not defined

In [ ]:
# VOICE ACTIVATED - opens on your voice, stays hot while you are talking, drops the
# socket once you go quiet. Needs cell 0 only.
import json, time, collections, httpx
from array import array

THRESHOLD  = 0.010  # mic RMS counting as speech. MacBook mic: room noise 0.002, speech ~0.03
PREROLL    = 0.6    # seconds kept before the trigger so your first word is not clipped
IDLE_CLOSE = 45     # seconds of quiet before the socket is dropped again
MIC        = "MacBook"  # substring of input device. AirPods mic reads pure silence on this
                        # machine so it would never trigger. None = system default.
MODEL      = "gpt-oss-120b"
SYSTEM     = ("Respond shortly. You are a voice assistant and your reply is read aloud: "
              "answer in one or two plain spoken sentences. Never use markdown, lists, "
              "bullets, asterisks, headings, or emoji. Write numbers and units as words.")

llm  = httpx.Client(base_url="https://api.cerebras.ai/v1", timeout=30,
                    headers={"Authorization": f"Bearer {os.environ['CEREBUS']}"})
chat = [{"role": "system", "content": SYSTEM}]
mic  = queue.Queue()
flushed, speaking, MIC_RATE = 0, False, RATE

_talk_stack = contextlib.ExitStack()
talk = _talk_stack.enter_context(
    dg.speak.v1.connect(model="aura-2-jupiter-en", encoding="linear16",
                        sample_rate=24000, speed=1.2))

def _talk_pump():
    global flushed
    try:
        for m in talk:
            if isinstance(m, bytes):
                play(m)
            elif getattr(m, "type", None) == "Flushed":
                flushed += 1
    except Exception:
        pass
threading.Thread(target=_talk_pump, daemon=True).start()

def rms(block):                       # audioop was removed in Python 3.13
    a = array("h"); a.frombytes(block)
    return (sum(s * s for s in a) / len(a)) ** 0.5 / 32768

def respond(heard):
    global speaking
    chat.append({"role": "user", "content": heard})
    speaking = True
    before, n, answer = flushed, 0, ""
    with llm.stream("POST", "/chat/completions", json={
            "model": MODEL, "messages": chat, "stream": True,
            "reasoning_effort": "low", "max_completion_tokens": 400,
            "temperature": 0.6}) as r:
        if r.status_code != 200:
            print(f"  [cerebras {r.status_code}] {r.read().decode()[:140]}")
            speaking = False
            return
        for line in r.iter_lines():
            if not line.startswith("data: "):
                continue
            body = line[6:]
            if body == "[DONE]":
                break
            tok = json.loads(body)["choices"][0]["delta"].get("content") or ""
            if not tok:
                continue
            answer += tok
            talk.send_text(SpeakV1Text(type="Speak", text=tok))
            if tok.rstrip()[-1:] in ".!?":
                talk.send_flush(); n += 1
    talk.send_flush(); n += 1
    chat.append({"role": "assistant", "content": answer})
    print(f"  jarvis: {answer.strip() or '[empty - reasoning used the whole budget]'}")
    end = time.time() + 30
    while flushed - before < n and time.time() < end:
        time.sleep(0.02)
    time.sleep(0.25)
    speaking = False

def session(preroll):
    """One connection, held open across as many turns as you keep talking."""
    stop, last = threading.Event(), [time.time()]
    with dg.listen.v2.connect(model="flux-general-en", encoding="linear16",
                              sample_rate=MIC_RATE, eager_eot_threshold=0.3,
                              eot_threshold=0.5, eot_timeout_ms=2000,
                              keyterm=KEYTERMS) as sock:
        for _ in range(40):
            sock.send_media(bytes(MIC_RATE // 20 * 2))   # warm once per connection
        for b in preroll:
            sock.send_media(b)

        def feed():
            while not stop.is_set():
                try:
                    blk = mic.get(timeout=0.2)
                except queue.Empty:
                    if time.time() - last[0] > IDLE_CLOSE: break
                    continue
                if speaking:
                    blk = bytes(len(blk))          # mute itself, keep the socket alive
                elif rms(blk) > THRESHOLD:
                    last[0] = time.time()
                try:
                    sock.send_media(blk)
                except Exception:
                    break
                if time.time() - last[0] > IDLE_CLOSE: break
            stop.set()
            try: sock.send_close_stream()
            except Exception: pass

        t = threading.Thread(target=feed, daemon=True); t.start()
        try:
            for m in sock:
                if stop.is_set():
                    break
                ev = getattr(m, "event", None)
                if ev == "EndOfTurn" and m.transcript.strip():
                    print(f"you:    {m.transcript}".ljust(100))
                    respond(m.transcript)
                    while not mic.empty():
                        mic.get()                  # drop audio buffered while it spoke
                    last[0] = time.time()
                elif ev in ("StartOfTurn", "Update", "EagerEndOfTurn") and m.transcript:
                    print(f"you:    {m.transcript[:88]}".ljust(100), end="\r", flush=True)
        except Exception:
            pass
        finally:
            stop.set(); t.join(timeout=1)

def open_mic(match=MIC):
    """Re-scan devices, then open. A kernel that has cycled streams (or seen Bluetooth
    come and go) leaves PortAudio stale, which shows up as 'Internal PortAudio error
    [-9986]'; the rescan plus fallbacks is what clears it."""
    global MIC_RATE
    sd._terminate(); sd._initialize()
    ins = [(i, d) for i, d in enumerate(sd.query_devices()) if d["max_input_channels"] > 0]
    if match:
        ins = [(i, d) for i, d in ins if match.lower() in d["name"].lower()] or ins
    else:
        ins.sort(key=lambda t: t[0] != sd.default.device[0])
    last_err = None
    for i, d in ins:
        for sr in dict.fromkeys([int(d["default_samplerate"]), 48000, 24000, 16000]):
            try:
                st = sd.RawInputStream(samplerate=sr, channels=1, dtype="int16",
                                       blocksize=sr // 20, device=i,
                                       callback=lambda b, *_: mic.put(bytes(b)))
                st.start()
                MIC_RATE = sr
                print(f"mic: {d['name']} @ {sr}")
                return st
            except Exception as e:
                last_err = e
    raise RuntimeError(f"could not open any input device. Last error: {last_err}")

pre = collections.deque(maxlen=int(PREROLL * 20))
try:
    with open_mic():
        print(f"idle - speak to connect. Socket drops after {IDLE_CLOSE}s quiet.\n")
        while True:
            b = mic.get()
            if speaking:
                pre.clear(); continue
            pre.append(b)
            if rms(b) > THRESHOLD:
                session(list(pre))                 # hot for the whole conversation
                pre.clear()
                while not mic.empty():
                    mic.get()
                print("idle\n")
except KeyboardInterrupt:
    pass

mic: MacBook Pro Microphone @ 48000
idle - speak to connect. Socket drops after 45s quiet.

you:    Hello?                                                                                      
  jarvis: Yes, I'm here. How can I help you today?
you:    Baskin, you respond to this.                                                                
  jarvis: Sure thing—what would you like to talk about?
you:    Hello.                                                                                      
  jarvis: Hello! What can I do for you?
you:    Oakland.                                                                                    
  jarvis: Oakland—what would you like to know about it?
you:    Auckland.                                                                                   
  jarvis: Auckland—what would you like to discuss?
you:    Tammy development.                                                                          
  jarvis: Tammy’s development—what specific info

: 